In [ ]:
### Method 1 ###
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
ir_df = pd.read_csv('CPI_PPI_2001_2022.csv')
ir_df.head()

In [ ]:
# Date period: from 2001.01 to 2022.07
y_cpi = ir_df['CPI'].astype(float)
y_ppi = ir_df['PPI'].astype(float)
date = ir_df['Date'].astype(str)

date
month_year = date

y_cpi

In [ ]:
### Now time for ARIMA prediction of time series, PAST predicts FUTURE!!!??? ###
### ARIMA = Auto Regressive Integrated Moving Average ###
### ARIMA requires dataset to be stationary; which means time series data does not vary over time, particularly mean, variance and covariance remain unchanged/constant with time! ###
# need to determine ARIMA parameter values: p, d, q #
# p: the order of the AR term
# d: the number of differencing
# q: the order of the MA term
### Use statistical testing (ADF - Augmented Dickey-Fuller and KPSS - Kwiatkowski-Phillips-Schmidt-Shin) to test for stationarity ###
### ADF uses differencing to transform the dataset; KPSS removes the trend to make the data stationary ###

from statsmodels.tsa.stattools import adfuller

adf_test = adfuller(y_cpi)
print('stat=%.3f, p=%.3f,' %adf_test[0:2])

if adf_test[1] > 0.05:
    print('Probably not stationary!')
else:
    print('Probably stationary!')

In [ ]:
from statsmodels.tsa.stattools import kpss

kpss_test = kpss(y_cpi, nlags='auto')
print('stat=%.3f, p=%.3f,' %kpss_test[0:2])

if kpss_test[1] > 0.05:
    print('Probably stationary!')
else:
    print('Probably not stationary!')

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

y_cpi_list = y_cpi
result = seasonal_decompose(y_cpi_list, model='additive', period=1)

result.plot()

In [ ]:
### After making dataset stationary, now moving onto determining the p,d,q for the ARIMA model ###

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

date_fx_log_diff = y_cpi_list.values.reshape(-1)

plot_acf(date_fx_log_diff, lags=50)
plot_pacf(date_fx_log_diff, lags=50)

### plot.show()

In [ ]:
### ARIMA Forecasting ###

from statsmodels.tsa.arima.model import ARIMA

#y = y_cpi.iloc[:, 1]
y = y_cpi

y

In [ ]:
# try different combinations of p,d,q parameters
model_arima = ARIMA(y, order=(4,0,2))
model_arima_fit = model_arima.fit()
print(model_arima_fit.summary())

In [ ]:
# try to find the best combinations of p,d,q parameters to give lowest AIC value and best result for ARIMA model; also making sure ar/ma p values < 0.05
# AIC (Akaike Information Criteria) is widely used measure of a statistical model. It quantifies 1) the goodness of fit, and 2) the simplicity/parsimony, of the model.
# p found in ACF (AR - own lagged values), q found in PACF (MA - error terms), d how many times the series has been differenced to achieve stationarity

import itertools

p = range(1, 5)
d = range(0, 2)
q = range(1, 7)

pdq = list(itertools.product(p,d,q))

aics = []
params = []

for param in pdq:
    model = ARIMA(y, order=param)
    model_fit = model.fit()
    aic = model_fit.aic
    aics.append(aic)
    params.append(param)
    
#combo = list(zip(aics, params))
#combo.sort()

#combo_array = np.array(combo)
#print(combo_array)
combo = [(aic, str(param)) for aic, param in zip(aics, params)]
combo.sort()

combo_array = np.array(combo, dtype=object)  # Use dtype=object for mixed types
print(combo_array)


In [ ]:
pred = model_arima_fit.forecast(5, alpha=0.05)

print(pred.values)

forecast = pred.tolist()

forecast

In [ ]:
forecast

In [ ]:
month_year_future = ['2022/8/1', '2022/9/1', '2022/10/1', '2022/11/1', '2022/12/1']

fx_2022 = np.array(list(zip(month_year_future, forecast)))
print(fx_2022)

In [ ]:
x_merge = ['2022/7/1', '2022/8/1']
y_merge = [y_cpi[-1:], forecast[0]]

month_year

In [ ]:
y_cpi[-1:]

In [ ]:
plt.figure(figsize=(14,6), facecolor='white')
plt.plot(month_year, y_cpi, color='purple')
#plt.plot(x_merge, y_merge, color='y')
plt.plot(month_year_future, forecast, color='y')
plt.legend(['Original CPI data', 'Forecast'])
plt.title('ARIMA: CPI Growth Rate')
plt.xlabel('Month-Year')
plt.ylabel('CPI Growth Rate')
plt.show()

In [ ]:
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

x_merge = ['2022/7/1', '2022/8/1']
y_merge = [y_cpi[-1:], forecast[0]]

plt.figure(figsize=(14,6), facecolor='white')
plt.plot(month_year, y_cpi, color='purple')
#plt.plot(x_merge, y_merge, color='y')
plt.plot(month_year_future, forecast, color='y')
plt.legend(['原始CPI数据', 'CPI同比预测'])
plt.title('ARIMA: 消费者价格指数同比增长率')
plt.xlabel('年-月')
plt.ylabel('CPI同比增长率')
plt.show()

In [ ]:
import matplotlib.dates as mdates

date = pd.to_datetime(ir_df['Date'], format='%Y/%m/%d')
month_year = date

x_merge_new = pd.to_datetime(x_merge, format='%Y/%m/%d')
date_new = pd.to_datetime(month_year_future, format='%Y/%m/%d')

plt.figure(figsize=(16,8), facecolor='white')
plt.plot(month_year, y_cpi, color='purple')
#plt.plot(x_merge_new, y_merge, color='y')
plt.plot(date_new, forecast, color='y')

ax = plt.gca()
#set monthly locator
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=12))
#set formatter
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
#set font and rotation for date tick labels
plt.gcf().autofmt_xdate()
plt.legend(['原始CPI数据','CPI同比预测'])
plt.title('ARIMA: 消费者价格指数同比增长率')
plt.xlabel('年-月')
plt.ylabel('CPI同比增长率')
plt.grid(True)

plt.savefig('CPI_中国.png', bbox_inches='tight')
plt.show()

In [ ]:
### Method 2: using pmdarima package to find best p,q values (4,0,2) ###
import pmdarima as pm
ARIMA_model = pm.auto_arima(y, start_p=1, start_q=1, test='adf', max_p=4, max_q=7,
                           m=1, #frequency of series, if m==1, seasonal is set to False automatically
                           d=None, #let model determine 'd' difference value
                           seasonal=False,
                           trace=True, #logs
                           error_action='warn', #show errors
                           suppress_warnings=True,
                           stepwise=True)


In [ ]:
ARIMA_model.plot_diagnostics(figsize=(15,12))
plt.show()

In [ ]:
pred_arima = ARIMA_model.predict(12, alpha=0.05)
print(pred_arima.values)

forecast_arima = pred_arima.tolist()
forecast_arima

In [ ]:
# Try SARIMA seaonal model - 36 months
SARIMA_model = pm.auto_arima(y, start_p=1, start_q=1, test='adf', max_p=4, max_q=7,
                             m=36, #frequency of series, if m==1, seasonal is set to False automatically
                             start_P=0,
                             d=None, #let model determine 'd' difference value
                             seasonal=True,
                             D=1, #order of the seasonal differencing
                             trace=True, #logs
                             error_action='warn', #show errors
                             suppress_warnings=True,
                             stepwise=True)

In [ ]:
SARIMA_model.plot_diagnostics(figsize=(15,12))
plt.show()

In [ ]:
pred_sarima = SARIMA_model.predict(12, alpha=0.05)
print(pred_sarima.values)

forecast_sarima = pred_sarima.tolist()
forecast_sarima

In [ ]:
# Try SARIMA seaonal model - 24 months
SARIMA_model = pm.auto_arima(y, start_p=1, start_q=1, test='adf', max_p=4, max_q=7,
                             m=24, #frequency of series, if m==1, seasonal is set to False automatically
                             start_P=0,
                             d=None, #let model determine 'd' difference value
                             seasonal=True,
                             D=1, #order of the seasonal differencing
                             trace=True, #logs
                             error_action='warn', #show errors
                             suppress_warnings=True,
                             stepwise=True)

In [ ]:
SARIMA_model.plot_diagnostics(figsize=(15,12))
plt.show()

In [ ]:
pred_sarima = SARIMA_model.predict(12, alpha=0.05)
print(pred_sarima.values)

forecast_sarima = pred_sarima.tolist()
forecast_sarima

In [ ]:
# Try SARIMA seaonal model - 12 months
SARIMA_model = pm.auto_arima(y, start_p=1, start_q=1, test='adf', max_p=4, max_q=7,
                             m=12, #frequency of series, if m==1, seasonal is set to False automatically
                             start_P=0,
                             d=None, #let model determine 'd' difference value
                             seasonal=True,
                             D=1, #order of the seasonal differencing
                             trace=True, #logs
                             error_action='warn', #show errors
                             suppress_warnings=True,
                             stepwise=True)

In [ ]:
SARIMA_model.plot_diagnostics(figsize=(15,12))
plt.show()

In [ ]:
pred_sarima = SARIMA_model.predict(12, alpha=0.05)
print(pred_sarima.values)

forecast_sarima = pred_sarima.tolist()
forecast_sarima

In [ ]:
##########################

In [ ]:
y_ppi

In [ ]:
### predict PPI ###

### Method 2: using pmdarima package to find best p,q values (4,0,2) ###
import pmdarima as pm
ARIMA_model = pm.auto_arima(y_ppi, start_p=1, start_q=1, test='adf', max_p=4, max_q=7,
                           m=1, #frequency of series, if m==1, seasonal is set to False automatically
                           d=None, #let model determine 'd' difference value
                           seasonal=False,
                           trace=True, #logs
                           error_action='warn', #show errors
                           suppress_warnings=True,
                           stepwise=True)


In [ ]:
ARIMA_model.plot_diagnostics(figsize=(15,12))
plt.show()

In [ ]:
pred_arima = ARIMA_model.predict(12, alpha=0.05)
print(pred_arima.values)

forecast_arima = pred_arima.tolist()
forecast_arima

In [ ]:
# Try SARIMA seaonal model - 36 months
SARIMA_model = pm.auto_arima(y_ppi, start_p=1, start_q=1, test='adf', max_p=4, max_q=7,
                             m=36, #frequency of series, if m==1, seasonal is set to False automatically
                             start_P=0,
                             d=None, #let model determine 'd' difference value
                             seasonal=True,
                             D=1, #order of the seasonal differencing
                             trace=True, #logs
                             error_action='warn', #show errors
                             suppress_warnings=True,
                             stepwise=True)

In [ ]:
SARIMA_model.plot_diagnostics(figsize=(15,12))
plt.show()

In [ ]:
pred_sarima = SARIMA_model.predict(12, alpha=0.05)
print(pred_sarima.values)

forecast_sarima = pred_sarima.tolist()
forecast_sarima

In [ ]:
# Try SARIMA seaonal model - 24 months
SARIMA_model = pm.auto_arima(y_ppi, start_p=1, start_q=1, test='adf', max_p=4, max_q=7,
                             m=24, #frequency of series, if m==1, seasonal is set to False automatically
                             start_P=0,
                             d=None, #let model determine 'd' difference value
                             seasonal=True,
                             D=1, #order of the seasonal differencing
                             trace=True, #logs
                             error_action='warn', #show errors
                             suppress_warnings=True,
                             stepwise=True)

In [ ]:
SARIMA_model.plot_diagnostics(figsize=(15,12))
plt.show()

In [ ]:
pred_sarima = SARIMA_model.predict(12, alpha=0.05)
print(pred_sarima.values)

forecast_sarima = pred_sarima.tolist()
forecast_sarima

In [ ]:
# Try SARIMA seaonal model - 12 months
SARIMA_model = pm.auto_arima(y_ppi, start_p=1, start_q=1, test='adf', max_p=4, max_q=7,
                             m=12, #frequency of series, if m==1, seasonal is set to False automatically
                             start_P=0,
                             d=None, #let model determine 'd' difference value
                             seasonal=True,
                             D=1, #order of the seasonal differencing
                             trace=True, #logs
                             error_action='warn', #show errors
                             suppress_warnings=True,
                             stepwise=True)

In [ ]:
SARIMA_model.plot_diagnostics(figsize=(15,12))
plt.show()

In [ ]:
pred_sarima = SARIMA_model.predict(12, alpha=0.05)
print(pred_sarima.values)

forecast_sarima = pred_sarima.tolist()
forecast_sarima